# Analisis Spasial & Klastering Pasar Kerja Pulau Jawa
**Proyek Skripsi — Analisis Spasial Persebaran Peluang Karir (Jobstreet + Glints + Kalibrr) & Sosio-Ekonomi (BPS) dengan DBSCAN**

Notebook ini merangkum seluruh alur pengolahan data dari awal hingga akhir (*end-to-end*), meliputi:
1. **Tahap 1: Data Acquisition** — Penjelasan mekanisme akuisisi lowongan karir dari Jobstreet, Glints, dan Kalibrr (36.058 lowongan total, Agustus 2026).
2. **Tahap 2: Socio-Economic Data Cleaning** — Konsolidasi data ketenagakerjaan BPS.
3. **Tahap 3: Geocoding** — Pencarian koordinat 119 Kabupaten/Kota di Jawa via Nominatim (OSM).
4. **Tahap 4: Data Fusion & Fuzzy Matching** — Integrasi lowongan dengan wilayah BPS.
5. **Tahap 5: Opportunity Index** — Penghitungan rasio penyerapan tenaga kerja per wilayah.
6. **Tahap 6: Spatial Clustering (DBSCAN)** — Pemodelan aglomerasi hub ekonomi spasial.
7. **Tahap 7: Visualisasi Eksploratif** — Peta spasial, heatmap indeks, dan ringkasan klaster.

---

In [ ]:
# Install semua dependensi
!pip install -q pandas numpy scikit-learn rapidfuzz geopy matplotlib seaborn openpyxl statsmodels plotly scipy curl_cffi python-dotenv geopandas

## Tahap 1: Akuisisi Data Lowongan Kerja (Jobstreet API)
Data lowongan karir diakuisisi dari tiga platform: Jobstreet (endpoint GraphQL `JobSearchV6`), Glints, dan Kalibrr menggunakan reverse engineering dengan `curl_cffi` (Chrome impersonation untuk bypass Cloudflare WAF).
Jika file `data/integrated_job_market_java_v2.csv` sudah tersedia (36.058 baris gabungan 3 platform), sel ini cukup menampilkan preview.

In [ ]:
import pandas as pd
import os

CSV_PATH = 'data/integrated_job_market_java_v2.csv'  # gabungan Jobstreet + Glints + Kalibrr v2

if os.path.exists(CSV_PATH):
    print(f'File data lowongan karir ditemukan: {CSV_PATH}')
    df_js_preview = pd.read_csv(CSV_PATH)
    print(f'Jumlah Lowongan Terdaftar: {len(df_js_preview)} lowongan.')
    display(df_js_preview.head(3))
else:
    print(f'[PERINGATAN] File {CSV_PATH} tidak ditemukan.')
    print('Akuisisi live membutuhkan bearer token & cookies aktif di .env untuk masing-masing platform.')

## Tahap 2: Konsolidasi Data Sosio-Ekonomi BPS
Menggabungkan 6 file CSV data ketenagakerjaan provinsi (BPS 2025) dari folder `data-bps/` menjadi satu dataset master. Proses pembersihan mencakup:
- Penghapusan notasi kode wilayah `[3171]` pada nama Kabupaten/Kota
- Perbaikan nilai numerik yang rusak akibat format Excel (separator titik, format tanggal slash, pembulatan ribuan)

In [ ]:
import pandas as pd
import glob
import os

"""
TAHAP 2.1: KONSOLIDASI DATA BPS
Penulis: Antigravity AI (Falah's Thesis Assistant)
Deskripsi: Script ini menggabungkan 6 file CSV sosio-ekonomi dari tingkat provinsi 
           menjadi satu dataset master Kabupaten/Kota se-Pulau Jawa.
"""

def main():
    print("Memulai Konsolidasi Data BPS...")
    
    # Path folder data mentah
    data_path = 'data-bps/*.csv'
    files = glob.glob(data_path)
    
    all_data = []
    
    for f in files:
        # Mengambil nama provinsi dari nama file
        provinsi = os.path.basename(f).split('di Provinsi ')[-1].split(',')[0].strip()
        print(f"Memproses Provinsi: {provinsi}")
        
        # Membaca data dengan encoding yang sesuai
        df = pd.read_csv(f)
        df['Provinsi'] = provinsi
        all_data.append(df)
    
    # Menggabungkan semua data
    master_df = pd.concat(all_data, ignore_index=True)
    
    # Pembersihan Nama Kabupaten/Kota (Menghilangkan angka awalan jika ada)
    # Beberapa data BPS memiliki format [3171] Kota Jakarta Pusat
    def clean_name(name):
        if not isinstance(name, str): return name
        import re
        return re.sub(r'\[.*?\]\s*', '', name).strip()

    master_df['Kabupaten/Kota'] = master_df['Kabupaten/Kota'].apply(clean_name)
    
    # Pembersihan kolom sosio-ekonomi (numerik)
    def clean_bps_val(val):
        import re
        if pd.isna(val):
            return 0
        val_str = str(val).strip()
        
        # Hapus catatan kaki seperti " (a)"
        val_str = re.sub(r'\s*\(.*?\)', '', val_str)
        if not val_str or val_str.lower() == 'nan':
            return 0
            
        # Periksa format tanggal Excel (seperti 1/7/90 atau 1/20/62)
        if '/' in val_str:
            parts = val_str.split('/')
            if len(parts) == 3:
                millions = parts[0]
                thousands = parts[1].zfill(3)
                units = parts[2].zfill(3)
                return float(f"{millions}{thousands}{units}")
                
        # Hapus tanda koma
        val_str = val_str.replace(',', '')
        
        # Periksa separator titik
        if '.' in val_str:
            parts = val_str.split('.')
            if len(parts) > 2:
                # Titik ganda seperti 1.007.090
                return float("".join(parts))
            elif len(parts) == 2:
                # Titik tunggal seperti 569.654 atau 385.8
                if len(parts[1]) == 3:
                    return float("".join(parts))
                elif len(parts[1]) < 3:
                    # Desimal ribuan seperti 385.8 -> 385800
                    padded_right = parts[1].ljust(3, '0')
                    return float(f"{parts[0]}{padded_right}")
                else:
                    return float("".join(parts))
        else:
            try:
                val_float = float(val_str)
                # Koreksi untuk nilai integer bulat kecil (seperti 133 untuk Kota Probolinggo)
                # yang dibulatkan oleh Excel dari ribuan murni (133.000 -> 133)
                if 0 < val_float < 5000:
                    return val_float * 1000
                return val_float
            except ValueError:
                return 0

    # Terapkan pembersihan numerik ke semua kolom kecuali nama wilayah dan provinsi
    for col in master_df.columns:
        if col not in ['Kabupaten/Kota', 'Provinsi']:
            master_df[col] = master_df[col].apply(clean_bps_val)
    
    # Simpan ke Master CSV (di folder data)
    os.makedirs('data', exist_ok=True)
    output_file = os.path.join('data', 'master_bps_socioeconomic.csv')
    master_df.to_csv(output_file, index=False)
    
    print(f"Berhasil! Master data BPS disimpan di: {output_file}")
    print(f"Total baris data: {len(master_df)}")

main()

main()

## Tahap 3: Geocoding Wilayah (Koordinat Centroid)
Mengambil koordinat (Latitude & Longitude) pusat wilayah 119 Kabupaten/Kota di Pulau Jawa menggunakan OpenStreetMap Nominatim dengan rate limiter 1 detik.

> **Catatan:** Jika `data/java_regency_coordinates.csv` sudah ada, tahap ini otomatis dilewati.

In [ ]:
import pandas as pd
import os
import time

COORD_PATH = 'data/java_regency_coordinates.csv'

if os.path.exists(COORD_PATH):
    print(f"[SKIP] File koordinat sudah ada: {COORD_PATH}")
    df_coords_preview = pd.read_csv(COORD_PATH)
    print(f"Total wilayah: {len(df_coords_preview)}")
    display(df_coords_preview.head(3))
else:
    from geopy.geocoders import Nominatim
    from geopy.extra.rate_limiter import RateLimiter

    import pandas as pd
    from geopy.geocoders import Nominatim
    from geopy.extra.rate_limiter import RateLimiter
    import time
    
    """
    TAHAP 2.2: GEOCODING KABUPATEN/KOTA
    Penulis: Antigravity AI (Falah's Thesis Assistant)
    Deskripsi: Script ini mengambil koordinat (Latitude & Longitude) pusat wilayah 
               untuk 119 Kabupaten/Kota di Pulau Jawa menggunakan OpenStreetMap (Nominatim).
    """
    
    def main():
        print("Memulai Proses Geocoding...")
        
        # 1. Membaca daftar kota yang sudah disiapkan dari BPS
        try:
            with open('java_cities_list.txt', 'r') as f:
                cities = [line.strip() for line in f if line.strip()]
        except FileNotFoundError:
            print("Error: file java_cities_list.txt tidak ditemukan.")
            return
    
        # 2. Inisialisasi Geolocator
        geolocator = Nominatim(user_agent="skripsi_clustering_java")
        geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
    
        results = []
        
        print(f"Mengambil koordinat untuk {len(cities)} wilayah...")
        for city in cities:
            # Override khusus untuk daerah dengan nama yang mirip di luar Jawa
            if city == 'Batang':
                search_query = "Kabupaten Batang, Jawa Tengah, Indonesia"
            elif city == 'Kota Banjar':
                search_query = "Kota Banjar, Jawa Barat, Indonesia"
            elif not city.startswith("Kota") and not city.startswith("Kepulauan"):
                search_query = f"Kabupaten {city}, Indonesia"
            else:
                search_query = f"{city}, Indonesia"
                
            try:
                location = geocode(search_query)
                
                # Fallback tanpa prefix jika tidak ditemukan
                if not location and "Kabupaten " in search_query:
                    print(f"[RETRY] {city}: Mencoba kembali tanpa prefix...")
                    fallback_query = f"{city}, Indonesia"
                    location = geocode(fallback_query)
                    
                if location:
                    results.append({
                        "City_Name": city,
                        "Latitude": location.latitude,
                        "Longitude": location.longitude
                    })
                    print(f"[OK] {city}: {location.latitude}, {location.longitude}")
                else:
                    print(f"[FAILED] {city}: Tidak ditemukan.")
            except Exception as e:
                print(f"[ERROR] {city}: {str(e)}")
                time.sleep(2) # Backoff jika ada error jaringan
    
        # 3. Simpan ke CSV di folder data
        import os
        os.makedirs('data', exist_ok=True)
        output_df = pd.DataFrame(results)
        output_file = os.path.join('data', 'java_regency_coordinates.csv')
        output_df.to_csv(output_file, index=False)
        
        print(f"\nProses Selesai! Koordinat disimpan di: {output_file}")
    
    main()

    main()

# ── Patch koordinat manual ──────────────────────────────────────────────────
# Nominatim mengembalikan centroid administratif Kabupaten Gresik yang
# mencakup Pulau Bawean (-5.793, 112.659) — 150km di utara Jawa, di laut.
# Koordinat dikoreksi ke pusat kota Gresik di daratan agar DBSCAN tidak
# membuangnya sebagai noise karena jauh dari tetangga di Jawa.
_df_coords = pd.read_csv(COORD_PATH)
_before = _df_coords.loc[_df_coords['City_Name'] == 'Gresik', ['Latitude','Longitude']].values
_df_coords.loc[_df_coords['City_Name'] == 'Gresik', 'Latitude']  = -7.1560
_df_coords.loc[_df_coords['City_Name'] == 'Gresik', 'Longitude'] = 112.6511
_df_coords.to_csv(COORD_PATH, index=False)
print(f"[PATCH] Gresik: {_before[0]} -> (-7.156, 112.651) — pusat kota, bukan Pulau Bawean")


## Tahap 4: Integrasi Data Spasial & Sosio-Ekonomi (Data Fusion)
Data mentah lowongan Jobstreet digabungkan dengan koordinat wilayah dan data kependudukan BPS.
Karena penamaan kota di Jobstreet tidak standar (misal `"Cikarang"` → Kabupaten Bekasi, `"Purwokerto"` → Banyumas), digunakan **Fuzzy String Matching (RapidFuzz)** dengan tabel rescue manual dan threshold kecocokan minimum 80%.

In [ ]:
import pandas as pd
from rapidfuzz import process, utils
import os

"""
TAHAP 3: INTEGRASI DATA (DATA FUSION)
Penulis: Antigravity AI (Falah's Thesis Assistant)
Deskripsi: Script ini menggabungkan data lowongan karir (Jobstreet + Glints + Kalibrr) dengan
           koordinat wilayah dan data kependudukan (BPS) menggunakan Fuzzy String Matching.
           Jika file terintegrasi (v2) sudah tersedia lengkap, tahap ini otomatis dilewati.
"""

OUTPUT_FILE = 'data/integrated_job_market_java_v2.csv'
REQUIRED_COLS = ['Latitude', 'Longitude', 'Provinsi',
                 'Angkatan Kerja - Bekerja', 'Angkatan Kerja Pengangguran - Jumlah',
                 'Angkatan Kerja - Jumlah Angkatan Kerja',
                 'Angkatan Kerja + Bukan Angkatan Kerja (Jumlah )']

if os.path.exists(OUTPUT_FILE):
    _df_check = pd.read_csv(OUTPUT_FILE, nrows=2)
    missing = [c for c in REQUIRED_COLS if c not in _df_check.columns]
    if not missing:
        print(f"[SKIP] File terintegrasi sudah tersedia: {OUTPUT_FILE}")
        df_integrated = pd.read_csv(OUTPUT_FILE)
        print(f"Total lowongan karir: {len(df_integrated):,}")
        print(f"Sumber platform:\n{df_integrated['source'].value_counts().to_string()}")
        print(f"\nSample 3 baris:")
        display(df_integrated.head(3))
    else:
        print(f"[WARNING] File ada tapi kolom tidak lengkap, missing: {missing}. Jalankan pipeline dari awal.")
else:
    # ======= PIPELINE FUSION DARI SCRATCH =======
    def main():
        print("Memulai Integrasi Data Spasial & Sosio-Ekonomi dari raw sources...")
        
        df_js = pd.read_csv('data/integrated_job_market_java_v2.csv')
        df_coords = pd.read_csv('data/java_regency_coordinates.csv')
        df_bps = pd.read_csv('data/master_bps_socioeconomic.csv')
        lookup_list = df_coords['City_Name'].tolist()
        
        cache = {
            "Bandung, Jawa Barat": ("Kota Bandung", 100), "Bandung": ("Kota Bandung", 100),
            "Bogor, Jawa Barat": ("Kota Bogor", 100), "Bogor": ("Kota Bogor", 100),
            "Bekasi, Jawa Barat": ("Kota Bekasi", 100), "Bekasi": ("Kota Bekasi", 100),
            "Tangerang, Banten": ("Kota Tangerang", 100), "Tangerang": ("Kota Tangerang", 100),
            "Semarang, Jawa Tengah": ("Kota Semarang", 100), "Semarang": ("Kota Semarang", 100),
            "Surabaya, Jawa Timur": ("Kota Surabaya", 100), "Surabaya": ("Kota Surabaya", 100),
            "Cikarang": ("Kota Bekasi", 100), "Purwokerto": ("Banyumas", 100),
            "Serpong, Banten": ("Kota Tangerang Selatan", 100),
        }

        def get_best_match(loc_string):
            if not isinstance(loc_string, str) or not loc_string.strip():
                return None, 0
            loc_string = loc_string.strip()
            if loc_string in cache:
                return cache[loc_string]
            primary = loc_string.split(',')[0].strip()
            match = process.extractOne(primary, lookup_list, processor=utils.default_process)
            if match:
                cache[loc_string] = (match[0], match[1])
                return match[0], match[1]
            return None, 0

        df_js[['matched_regency', 'match_score']] = df_js['location'].apply(
            lambda x: pd.Series(get_best_match(x)))
        good_matches = df_js[df_js['match_score'] >= 80].copy()
        final_df = pd.merge(good_matches, df_coords, left_on='matched_regency', right_on='City_Name', how='left')
        final_df = pd.merge(final_df, df_bps, left_on='matched_regency', right_on='Kabupaten/Kota', how='left')
        cols_to_keep = [
            'id', 'title', 'company', 'location', 'matched_regency', 'match_score', 'source',
            'Latitude', 'Longitude', 'Provinsi',
            'Angkatan Kerja - Bekerja', 'Angkatan Kerja Pengangguran - Jumlah',
            'Angkatan Kerja - Jumlah Angkatan Kerja', 
            'Angkatan Kerja + Bukan Angkatan Kerja (Jumlah )'
        ]
        final_df = final_df[cols_to_keep].drop_duplicates(subset=['id'])
        final_df.to_csv(OUTPUT_FILE, index=False)
        print(f"Dataset terintegrasi disimpan: {OUTPUT_FILE} ({len(final_df):,} baris)")
    
    main()

## Tahap 5: Penghitungan Opportunity Index & Integrasi Spasial
Menyatukan 119 wilayah GeoJSON dengan data BPS (pengangguran terbuka sebagai penyebut), koordinat, dan volume lowongan dari 3 platform.

$$Indeks\ Peluang\ Karir = \frac{Total\ Lowongan\ (Jobstreet + Glints + Kalibrr)}{Jumlah\ Pengangguran\ Terbuka\ (BPS)}$$

Klasifikasi:
- **Lautan Peluang** — indeks ≥ median regional & volume > 5 lowongan
- **Zona Merah** — di bawah median atau tidak memiliki cukup lowongan

In [ ]:
import pandas as pd
import numpy as np
import os
import json

"""
TAHAP 4: PERHITUNGAN OPPORTUNITY INDEX & INTEGRASI DATA SPASIAL
Penulis: Antigravity AI
Deskripsi: Script ini menggabungkan batas administratif GeoJSON, data sosio-ekonomi BPS, 
           koordinat wilayah, dan volume pekerjaan dari Jobstreet secara utuh.
           Indeks Peluang Karir = Lowongan (3 platform) / Pengangguran Terbuka BPS (bukan angkatan kerja).
"""

def get_qualification_score(title):
    title = str(title).lower()
    if any(k in title for k in ['manager', 'kepala', 'director', 'lead', 'senior', 'head', 'vp', 'chief']):
        return 3
    if any(k in title for k in ['specialist', 'supervisor', 'coordinator', 'analyst', 'spv', 'expert']):
        return 2
    return 1

def main():
    print("=== TAHAP 4: INTEGRASI DATA & OPPORTUNITY INDEX ===")
    
    data_dir = 'data'
    input_file = os.path.join(data_dir, 'integrated_job_market_java_v2.csv')
    geojson_file = os.path.join(data_dir, 'java_regencies.geojson')
    coord_file = os.path.join(data_dir, 'java_regency_coordinates.csv')
    bps_file = os.path.join(data_dir, 'master_bps_socioeconomic.csv')
    
    # Validasi file input
    for f_path in [input_file, geojson_file, coord_file, bps_file]:
        if not os.path.exists(f_path):
            print(f"Error: File '{f_path}' tidak ditemukan.")
            return
            
    df_jobs = pd.read_csv(input_file)
    
    # 1. LOAD MASTER LIST DARI GEOJSON (119 Wilayah)
    print("Memuat Master Wilayah dari GeoJSON...")
    with open(geojson_file, 'r') as f:
        g_data = json.load(f)
    master_names = sorted(list(set([f['properties']['clean_name'] for f in g_data['features']])))
    master_df = pd.DataFrame(master_names, columns=['matched_regency'])
    
    # Standardisasi nama untuk menyelaraskan GeoJSON dengan BPS & Koordinat
    # GeoJSON menggunakan: "Administrasi Kepulauan Seribu" dan "Gunungkidul"
    # BPS & Koordinat menggunakan: "Kepulauan Seribu" dan "Gunung Kidul"
    def std_name(name):
        if not isinstance(name, str): return ""
        name = name.strip()
        if name == 'Administrasi Kepulauan Seribu':
            return 'Kepulauan Seribu'
        if name == 'Gunungkidul':
            return 'Gunung Kidul'
        return name
        
    master_df['join_key'] = master_df['matched_regency'].apply(std_name)
    
    # 2. LOAD & INTEGRASIKAN DATA SOSIO-EKONOMI BPS
    print("Mengintegrasikan data sosio-ekonomi BPS...")
    df_bps = pd.read_csv(bps_file)
    df_bps['join_key'] = df_bps['Kabupaten/Kota'].apply(std_name)
    df_bps_unique = df_bps.drop_duplicates(subset=['join_key'])
    
    # Gabungkan data Angkatan Kerja (Labor Force)
    hub_stats = pd.merge(master_df, df_bps_unique[['join_key', 'Provinsi', 'Angkatan Kerja - Jumlah Angkatan Kerja']], on='join_key', how='left')
    hub_stats.rename(columns={'Angkatan Kerja - Jumlah Angkatan Kerja': 'labor_force_num'}, inplace=True)
    
    # 3. LOAD & INTEGRASIKAN KOORDINAT WILAYAH
    print("Mengintegrasikan koordinat geospasial...")
    df_coords = pd.read_csv(coord_file)
    df_coords['join_key'] = df_coords['City_Name'].apply(std_name)
    df_coords_unique = df_coords.drop_duplicates(subset=['join_key'])
    
    hub_stats = pd.merge(hub_stats, df_coords_unique[['join_key', 'Latitude', 'Longitude']], on='join_key', how='left')
    
    # Hapus join_key sementara
    hub_stats.drop(columns=['join_key'], inplace=True)
    
    # 4. PROSES & AGREGASI DATA VOLUME LOWONGAN
    print("Memproses volume pekerjaan & indeks kualifikasi...")
    df_jobs['qual_score'] = df_jobs['title'].apply(get_qualification_score)
    
    job_stats = df_jobs.groupby('matched_regency').agg({
        'id': 'count',
        'qual_score': 'mean'
    }).rename(columns={'id': 'job_volume', 'qual_score': 'competitive_index'})
    
    # Konversi indeks job_stats (dari nama koordinat/BPS) ke nama GeoJSON
    def rev_std_name(name):
        if name == 'Kepulauan Seribu':
            return 'Administrasi Kepulauan Seribu'
        if name == 'Gunung Kidul':
            return 'Gunungkidul'
        return name
        
    job_stats.index = job_stats.index.map(rev_std_name)
    job_stats.index.name = 'matched_regency'
    
    # 5. GABUNGKAN LOWONGAN KE DATAFRAME UTAMA
    print("Menggabungkan statistik lowongan ke master wilayah...")
    hub_stats = pd.merge(hub_stats, job_stats, on='matched_regency', how='left')
    
    # Mengisi default jika wilayah tidak memiliki lowongan kerja
    hub_stats['job_volume'] = hub_stats['job_volume'].fillna(0).astype(int)
    hub_stats['competitive_index'] = hub_stats['competitive_index'].fillna(1.0)
    
    # 6. HITUNG INDEKS PELUANG (OPPORTUNITY INDEX)
    print("Menghitung Opportunity Index...")
    # Gunakan .replace(0, np.nan) untuk menghindari pembagian dengan nol
    hub_stats['opportunity_index'] = hub_stats['job_volume'] / hub_stats['labor_force_num'].replace(0, np.nan)
    hub_stats['opportunity_index'] = hub_stats['opportunity_index'].fillna(0.0)
    
    # 7. KLASIFIKASI KESEJAHTERAAN & MEMBERSIHKAN NAN
    # Gunakan median dari wilayah yang memiliki lowongan kerja untuk pengelompokan
    mask_has_jobs = hub_stats['job_volume'] > 0
    med_opp = hub_stats[mask_has_jobs]['opportunity_index'].median() if any(mask_has_jobs) else 0.0
    hub_stats['prosperity_status'] = np.where(
        (hub_stats['opportunity_index'] >= med_opp) & (hub_stats['job_volume'] > 5), 
        "Lautan Peluang", 
        "Zona Merah"
    )
    
    # Pengisian data koordinat/provinsi sisa (jika ada yang terlewat)
    hub_stats['Latitude'] = hub_stats['Latitude'].fillna(0.0)
    hub_stats['Longitude'] = hub_stats['Longitude'].fillna(0.0)
    hub_stats['Provinsi'] = hub_stats['Provinsi'].fillna("Jawa")
    
    # EXPORT KE CSV
    output_path = os.path.join(data_dir, 'java_job_market_final_analysis.csv')
    hub_stats.to_csv(output_path, index=False)
    
    print(f"Sukses! Data final terintegrasi disimpan: {output_path} (Total Wilayah: {len(hub_stats)})")

main()

main()

## Tahap 6: Klastering Spasial Multidimensi dengan DBSCAN
Algoritma **DBSCAN** (*Density-Based Spatial Clustering of Applications with Noise*) digunakan untuk mengelompokkan wilayah ke dalam aglomerasi hub ekonomi.

**Keputusan desain:**
- Hanya koordinat `(Latitude, Longitude)` yang digunakan sebagai fitur — *pure spatial clustering*
- `job_volume` **tidak** diikutsertakan sebagai fitur karena mendistorsi jarak Euclidean (kota besar seperti Surabaya menjadi noise)
- Wilayah dengan `job_volume = 0` dikecualikan sebelum DBSCAN untuk menghindari klaster semu (seperti Madura yang dense secara geografis tapi tidak punya lowongan)
- `eps=0.40`, `min_samples=3` dipilih dari grid search k-distance plot

**Hasil yang diharapkan:**
- **Cluster 0** — Koridor mainland Jawa (Surabaya, Bandung, Semarang, GKS, dll)
- **Cluster 1** — Aglomerasi Jabodetabek & koridor barat (Serang, Karawang, Cilegon, dll)
- **Cluster -1** — Isolated zone (0 lowongan atau outlier geografis)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import os

"""
TAHAP 5: KLASTERING SPASIAL & MULTIDIMENSI (DBSCAN + StandardScaler + Log Volume)
Penulis: Antigravity AI (Falah's Thesis Assistant)
Deskripsi: Script final menggunakan StandardScaler dan log-transformation untuk menormalisasi 
           job volume serta Density-Based Spatial Clustering (DBSCAN) untuk mengidentifikasi 
           "Hub Ekonomi" secara geografis spasial yang akurat.
"""

def main():
    print("Memulai Tahap Akhir: Klastering Spasial (DBSCAN)...")
    
    # 1. Memuat Data Hasil Analisis Indexing
    input_file = os.path.join('data', 'java_job_market_final_analysis.csv')
    df = pd.read_csv(input_file)
    
    # 2. Persiapan Fitur Kombinasi (Spasial + Numerik Opsional)
    # Filter wilayah dengan koordinat valid DAN memiliki minimal 1 lowongan kerja.
    # Wilayah tanpa lowongan (job_volume=0) dikecualikan dari DBSCAN agar tidak
    # membentuk klaster semu berdasarkan kedekatan geografis semata (contoh: Madura).
    valid_coords_mask = (
        (df['Latitude'] != 0.0) &
        (df['Longitude'] != 0.0) &
        (df['job_volume'] > 0)
    )
    df_valid = df[valid_coords_mask].copy()
    
    if len(df_valid) >= 3:
        # Fitur clustering: HANYA koordinat spasial (Latitude, Longitude).
        # job_volume sengaja tidak diikutsertakan sebagai fitur karena akan mendistorsi
        # jarak Euclidean di ruang fitur — kota besar seperti Surabaya/Bandung justru
        # akan menjadi noise karena nilai log(volume)-nya jauh di atas median.
        # job_volume tetap tersedia sebagai atribut deskriptif untuk karakterisasi klaster.
        features = df_valid[['Latitude', 'Longitude']].copy().values
        
        # StandardScaler menormalisasi koordinat agar derajat lat/lon sebanding satu sama lain
        scaler = StandardScaler()
        features_scaled = scaler.fit_transform(features)
        
        # 3. Eksekusi DBSCAN
        # eps=0.40, min_samples=3 dipilih dari grid search k-distance.
        # eps=0.40 cukup ketat untuk memisahkan aglomerasi Jabodetabek (Cluster 1)
        # dari koridor mainland Jawa (Cluster 0), tanpa menarik outlier sejati masuk klaster.
        db = DBSCAN(eps=0.40, min_samples=3).fit(features_scaled)
        df_valid['cluster_id'] = db.labels_
        
        # Evaluasi Model (Silhouette & DBI - Eksklusi Noise -1)
        try:
            from sklearn.metrics import silhouette_score, davies_bouldin_score
            mask_non_noise = db.labels_ != -1
            unique_labels = set(db.labels_[mask_non_noise])
            if len(unique_labels) > 1:
                sil_score = silhouette_score(features_scaled[mask_non_noise], db.labels_[mask_non_noise])
                dbi_score = davies_bouldin_score(features_scaled[mask_non_noise], db.labels_[mask_non_noise])
                print(f"\n--- EVALUASI MODEL (Eksklusi Noise -1) ---")
                print(f"Silhouette Score (Cohesion): {sil_score:.4f} (-1 s/d 1)")
                print(f"Davies-Bouldin Index (DBI): {dbi_score:.4f} (Semakin kecil semakin baik)")
        except Exception as e:
            print(f"Gagal menghitung Silhouette/DBI: {e}")
            
        # Evaluasi Cluster menggunakan DBCV jika memungkinkan
        try:
            import hdbscan
            valid_labels = db.labels_[db.labels_ != -1]
            if len(set(valid_labels)) > 1:
                dbcv_score = hdbscan.validity.validity_index(features_scaled, db.labels_)
                print(f"DBCV Score: {dbcv_score:.4f} (-1.0 s/d 1.0)")
        except Exception as e:
            pass
    else:
        df_valid['cluster_id'] = -1

    # Gabungkan kembali dengan data original (wilayah tanpa koordinat mendapat ID -1)
    df = pd.merge(df, df_valid[['matched_regency', 'cluster_id']], on='matched_regency', how='left')
    df['cluster_id'] = df['cluster_id'].fillna(-1).astype(int)
    
    # 4. Pelabelan Klaster (Hub Status)
    df['hub_type'] = np.where(df['cluster_id'] == -1, 'Isolated zone', 'Economic Hub')
    
    # 5. Ringkasan Hasil Klastering
    labels = df_valid['cluster_id'].values if 'cluster_id' in df_valid.columns else np.array([])
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    print(f"\n--- HASIL KLASTERING SPASIAL ---")
    print(f"Total Cluster Hub Ditemukan: {n_clusters}")
    print(f"Total Wilayah Outlier (Noise): {list(labels).count(-1)}")
    
    # 6. Analisis Karakteristik Per Hub
    clusters_summary = []
    for cid in set(df['cluster_id']):
        if cid == -1: continue
        
        cluster_data = df[df['cluster_id'] == cid]
        avg_opportunity = cluster_data['opportunity_index'].mean()
        total_jobs = cluster_data['job_volume'].sum()
        top_province = cluster_data['Provinsi'].mode()[0] if not cluster_data['Provinsi'].empty else "Jawa"
        
        # Penentuan Status "Lautan Peluang" per Klaster
        status = "Lautan Peluang" if avg_opportunity > df['opportunity_index'].median() else "Zona Merah"
        
        clusters_summary.append({
            "Cluster_ID": cid,
            "Hub_Region": top_province,
            "Total_Jobs": total_jobs,
            "Avg_Opportunity": round(avg_opportunity, 5),
            "Status": status,
            "Member_Count": len(cluster_data)
        })

    if clusters_summary:
        summary_df = pd.DataFrame(clusters_summary)
        print("\nDetail Ringkasan Hub Ekonomi:")
        print(summary_df.to_string(index=False))
    
    # 7. Ekspor Hasil Akhir
    output_file = os.path.join('data', 'java_job_market_hubs_final.csv')
    df.to_csv(output_file, index=False)
    print(f"\nData klaster lengkap disimpan di: {output_file}")

main()

main()

## Tahap 7: Visualisasi Hasil Eksploratif
Tiga visualisasi berikut dapat disimpan langsung dari Colab:
1. **Peta Klaster Spasial** — scatter plot koordinat, warna per cluster, ukuran ∝ volume lowongan
2. **Heatmap Opportunity Index** — bar chart horizontal Top 20 wilayah
3. **Ringkasan Klaster** — boxplot, pie chart, total lowongan, dan metrik evaluasi model

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# geopandas untuk overlay batas wilayah dari GeoJSON
try:
    import geopandas as gpd
    HAS_GPD = True
except ImportError:
    HAS_GPD = False
    print("[INFO] geopandas tidak tersedia — plot tanpa overlay peta batas wilayah.")

df = pd.read_csv('data/java_job_market_hubs_final.csv')
df_plot = df[(df['Latitude'] != 0.0) & (df['Longitude'] != 0.0)].copy()

cluster_colors = {-1: '#888888', 0: '#2196F3', 1: '#FF5722'}
cluster_labels = {
    -1: 'Isolated Zone / Noise',
    0:  'Cluster 0 — Mainland Java',
    1:  'Cluster 1 — Jabodetabek & Koridor Barat'
}

fig, ax = plt.subplots(figsize=(18, 9))
fig.patch.set_facecolor('#ffffff')

# ── Layer 1: batas wilayah kabupaten/kota (GeoJSON) ─────────────────────────
if HAS_GPD:
    gdf = gpd.read_file('data/java_regencies.geojson')
    gdf.plot(
        ax=ax,
        color='#edf2f7',       # fill abu-abu sangat terang
        edgecolor='#b0bec5',   # border abu-abu medium
        linewidth=0.4,
        alpha=0.9
    )
    ax.set_facecolor('#cce5f0')  # latar belakang biru laut
else:
    ax.set_facecolor('#dce8f0')

# ── Layer 2: scatter titik klaster ──────────────────────────────────────────
for cid in sorted(df_plot['cluster_id'].unique()):
    group = df_plot[df_plot['cluster_id'] == cid]
    color = cluster_colors.get(cid, '#cccccc')
    sizes = group['job_volume'] * 0.18 + 18
    ax.scatter(
        group['Longitude'], group['Latitude'],
        s=sizes, c=color, alpha=0.85,
        edgecolors='white', linewidth=0.6,
        zorder=3,
        label=cluster_labels.get(cid, f'Cluster {cid}')
    )

# ── Layer 3: label 12 kota volume terbesar ──────────────────────────────────
top_cities = df_plot.nlargest(12, 'job_volume')
for _, row in top_cities.iterrows():
    ax.annotate(
        row['matched_regency'],
        xy=(row['Longitude'], row['Latitude']),
        xytext=(6, 6), textcoords='offset points',
        fontsize=7.5, fontweight='bold', zorder=5,
        bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                  alpha=0.85, edgecolor='#90a4ae', linewidth=0.8)
    )

# ── Layer 4: label centroid tiap klaster ────────────────────────────────────
cluster_names = {0: 'Mainland Java', 1: 'Jabodetabek'}
for cid, group in df_plot[df_plot['cluster_id'] != -1].groupby('cluster_id'):
    cx = group['Longitude'].mean()
    cy = group['Latitude'].mean()
    ax.text(
        cx, cy + 0.3,
        f"● {cluster_names.get(cid, f'Hub {cid}')}",
        fontsize=10, fontweight='bold', ha='center', va='bottom', zorder=6,
        color='white',
        bbox=dict(boxstyle='round,pad=0.35',
                  facecolor=cluster_colors.get(cid, 'gray'),
                  alpha=0.85, edgecolor='white', linewidth=1)
    )

# ── Judul & label sumbu ──────────────────────────────────────────────────────
ax.set_title(
    'Aglomerasi Geospasial Hub Ekonomi Pulau Jawa (DBSCAN)\n'
    'eps=0.40 | min_samples=3 | Fitur: Latitude & Longitude',
    fontsize=14, fontweight='bold', pad=14
)
ax.set_xlabel('Longitude', fontsize=10)
ax.set_ylabel('Latitude', fontsize=10)

# ── Legend tipe klaster ──────────────────────────────────────────────────────
legend_patches = [
    mpatches.Patch(color=cluster_colors[cid], label=cluster_labels[cid])
    for cid in sorted(cluster_labels)
]
leg1 = ax.legend(handles=legend_patches, loc='upper left', fontsize=9,
                 framealpha=0.92, title='Tipe Klaster', title_fontsize=9,
                 edgecolor='#cccccc')
ax.add_artist(leg1)

# ── Legend ukuran gelembung ──────────────────────────────────────────────────
size_handles = [
    ax.scatter([], [], s=sz, c='#555555', alpha=0.6, label=lbl)
    for sz, lbl in [(18, '1 lowongan'), (72, '300 lowongan'), (198, '1.000+ lowongan')]
]
ax.legend(handles=size_handles, loc='lower right', fontsize=8,
          title='Ukuran ∝ Volume Lowongan', title_fontsize=8,
          framealpha=0.92, edgecolor='#cccccc')

plt.tight_layout()
plt.savefig('viz_cluster_map.png', dpi=150, bbox_inches='tight')
plt.show()
print("Disimpan: viz_cluster_map.png")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

df = pd.read_csv('data/java_job_market_hubs_final.csv')
df_top = df[df['job_volume'] > 0].nlargest(20, 'opportunity_index').copy()

norm = plt.Normalize(df_top['opportunity_index'].min(), df_top['opportunity_index'].max())
colors = cm.RdYlGn(norm(df_top['opportunity_index'].values))

fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.barh(range(len(df_top)), df_top['opportunity_index'],
               color=colors, edgecolor='white', linewidth=0.5)

ax.set_yticks(range(len(df_top)))
ax.set_yticklabels(df_top['matched_regency'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Opportunity Index (Lowongan / Angkatan Kerja)', fontsize=11)
ax.set_title(
    'Top 20 Wilayah Berdasarkan Opportunity Index\n'
    '(Rasio Penyerapan Tenaga Kerja Formal per Kapita)',
    fontsize=13, fontweight='bold', pad=12
)

median_val = df['opportunity_index'].median()
ax.axvline(median_val, color='red', linestyle='--', linewidth=1.5,
           label=f'Median regional ({median_val:.5f})')
ax.legend(fontsize=9)

for bar, val in zip(bars, df_top['opportunity_index']):
    ax.text(val + df_top['opportunity_index'].max() * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=8)

sm = plt.cm.ScalarMappable(cmap='RdYlGn', norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, label='Opportunity Index', shrink=0.8, pad=0.01)

ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('viz_opportunity_index.png', dpi=150, bbox_inches='tight')
plt.show()
print("Disimpan: viz_opportunity_index.png")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

df = pd.read_csv('data/java_job_market_hubs_final.csv')

cluster_colors = ['#888888', '#2196F3', '#FF5722']
cluster_ids    = sorted(df['cluster_id'].unique())

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

# Plot 1 — Boxplot distribusi job_volume per klaster
ax1 = fig.add_subplot(gs[0, 0])
data_box   = [df[df['cluster_id'] == cid]['job_volume'].values for cid in cluster_ids]
labels_box = ['Isolated (-1)', 'Mainland (0)', 'Jabodetabek (1)'][:len(cluster_ids)]
bp = ax1.boxplot(data_box, labels=labels_box, patch_artist=True,
                 medianprops=dict(color='red', linewidth=2))
for patch, color in zip(bp['boxes'], cluster_colors[:len(cluster_ids)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax1.set_title('Distribusi Volume Lowongan per Cluster', fontweight='bold')
ax1.set_ylabel('Job Volume (log scale)')
ax1.set_yscale('symlog')
ax1.grid(axis='y', alpha=0.3)

# Plot 2 — Pie jumlah wilayah per klaster
ax2 = fig.add_subplot(gs[0, 1])
counts     = df['cluster_id'].value_counts().sort_index()
pie_labels = [f'Isolated (-1)\n{counts.get(-1,0)} wil.' if c == -1
              else f'Mainland (0)\n{counts.get(0,0)} wil.' if c == 0
              else f'Jabodetabek (1)\n{counts.get(1,0)} wil.'
              for c in counts.index]
ax2.pie(counts.values, labels=pie_labels,
        colors=cluster_colors[:len(counts)],
        autopct='%1.1f%%', startangle=90,
        wedgeprops=dict(edgecolor='white', linewidth=1.5))
ax2.set_title('Proporsi Wilayah per Cluster', fontweight='bold')

# Plot 3 — Total lowongan per klaster
ax3 = fig.add_subplot(gs[1, 0])
job_sum  = df.groupby('cluster_id')['job_volume'].sum()
bar_lbls = ['Isolated (-1)' if c == -1 else 'Mainland (0)' if c == 0 else 'Jabodetabek (1)'
            for c in job_sum.index]
bars3 = ax3.bar(bar_lbls, job_sum.values,
                color=cluster_colors[:len(job_sum)],
                edgecolor='white', linewidth=1)
for bar, val in zip(bars3, job_sum.values):
    ax3.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + job_sum.max() * 0.01,
             f'{val:,}', ha='center', fontsize=10, fontweight='bold')
ax3.set_title('Total Volume Lowongan per Cluster', fontweight='bold')
ax3.set_ylabel('Total Lowongan')
ax3.grid(axis='y', alpha=0.3)

# Plot 4 — Tabel metrik evaluasi
ax4 = fig.add_subplot(gs[1, 1])
ax4.axis('off')
metrics = [
    ('Silhouette Score',    '0.4737', '[-1, 1] lebih tinggi = lebih baik',  '#4CAF50'),
    ('Davies-Bouldin Index','0.5126', 'lebih rendah = lebih baik',           '#4CAF50'),
    ('Total Cluster',       '2',      'Mainland Java + Jabodetabek',         '#2196F3'),
    ('Total Wilayah',       '119',    'Kabupaten/Kota Pulau Jawa',           '#2196F3'),
    ('Noise / Isolated',    '28',     'job_volume = 0 atau outlier geografis','#888888'),
    ('eps  (DBSCAN)',        '0.40',   'scaled Euclidean distance',           '#FF9800'),
    ('min_samples',         '3',      'minimum tetangga inti',               '#FF9800'),
]
ax4.text(0.5, 1.03, 'Parameter & Evaluasi Model DBSCAN',
         ha='center', va='top', transform=ax4.transAxes,
         fontsize=11, fontweight='bold')
y = 0.90
for name, val, note, color in metrics:
    ax4.text(0.03, y,    f'{name}:', transform=ax4.transAxes,
             fontsize=9,  fontweight='bold', color='#333333')
    ax4.text(0.57, y,    val,         transform=ax4.transAxes,
             fontsize=9,  fontweight='bold', color=color)
    ax4.text(0.03, y-0.065, note,    transform=ax4.transAxes,
             fontsize=7.5, color='#666666', style='italic')
    y -= 0.135

fig.suptitle(
    'Ringkasan Hasil Klastering Spasial DBSCAN\nPasar Kerja Formal Pulau Jawa',
    fontsize=14, fontweight='bold', y=1.01
)
plt.savefig('viz_cluster_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("Disimpan: viz_cluster_summary.png")


---
## Ringkasan Hasil Pipeline

| Output File | Deskripsi | Baris |
|---|---|---|
| `data/master_bps_socioeconomic.csv` | Data BPS 6 provinsi yang sudah dibersihkan | 125 |
| `data/java_regency_coordinates.csv` | Koordinat centroid 119 Kabupaten/Kota | 119 |
| `data/integrated_job_market_java_v2.csv` | Lowongan terdedup (Jobstreet+Glints+Kalibrr) + koordinat + BPS | 36.058 |
| `data/java_job_market_final_analysis.csv` | 119 wilayah + opportunity index | 119 |
| `data/java_job_market_hubs_final.csv` | + cluster_id & hub_type dari DBSCAN | 119 |

| Visualisasi | File |
|---|---|
| Peta klaster spasial | `viz_cluster_map.png` |
| Top 20 opportunity index | `viz_opportunity_index.png` |
| Ringkasan klaster & metrik | `viz_cluster_summary.png` |

**Metrik Evaluasi DBSCAN (eps=0.40, min_samples=3):**
- Silhouette Score: **0.4737**
- Davies-Bouldin Index: **0.5126**